# MLIP examples: MoE Training and Inference


In this notebook, we present a minimal example of **how to train a mixture-of-experts (MoE) model with the *mlip* library**, and how to perform inference. The latter is done by contracting the parameters of the multiple experts trained using a set of weighting coefficients which are dependent on an `InferenceContext` (a set of embedding parameters: e.g. total charge, spin, dataset type).

For further information on the construction of MoE models in the context of MLIPs, we refer users to the [UMA](https://arxiv.org/abs/2506.23971) paper.

Note we keep this tutorial minimal. The goal is to demonstrate the mechanics of multi-dataset data processing, MoE routing, model saving, and subsequent efficient inference with a fixed context.

**This notebook aims at showcasing:**

- **How to prepare a multi-dataset training set** with `BuilderMode.MULTI`
- **How to create an eSEN Mixture-of-Experts model**
- **How to run MoE training** and save the trained model
- **How to load the MoE model with different contracted experts depending on the inference context**
- ***[Advanced topic]* How to route MoE models on charge and spin metadata**


**Install, required imports, and logging setup**


As a first step, we will run the installation of the *mlip* library directly from pip. We also install the appropriate Jax CUDA backend to run on GPU (comment it out to run on CPU). Note that if you have ran another tutorial in the same environment, this installation is not required. Please refer to [our installation page](https://instadeepai.github.io/mlip/installation/index.html) for more information.


In [ ]:
%pip install "mlip[cuda]" huggingface_hub

# Use this instead for installation without GPU:
# %pip install mlip huggingface_hub

In [ ]:
import logging
from pathlib import Path

import jax
from absl import logging as absl_logging
from ase.io import read as ase_read

from mlip.data import BuilderMode, ExtxyzReader, GraphDatasetBuilder
from mlip.inference import run_batched_inference
from mlip.models import Esen, ForceField, InferenceContext
from mlip.models.esen.config import EsenConfig, EsenMoEConfig
from mlip.models.loss import MSELoss
from mlip.models.model_io import load_model_from_zip, save_model_to_zip
from mlip.training import TrainingLoop, get_default_mlip_optimizer

logging.basicConfig(
    level=logging.INFO, force=True, format="%(levelname)s - %(message)s"
)
logging.getLogger("mlip").setLevel(logging.INFO)
absl_logging.set_verbosity(absl_logging.WARNING)

print(jax.devices())

## 1. Preparing a multi-dataset training set

For this example, we train on two small datasets: aspirin and 3BPA. The datasets are kept deliberately small so that this tutorial remains fast. We first download the data from huggingface.


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns="mixture_of_experts/*",
    local_dir=".",
)

The data processing is analogous to the [model training tutorial](https://github.com/instadeepai/mlip/blob/main/tutorials/model_training_tutorial.ipynb), but uses `BuilderMode.MULTI`. In this mode, the readers dictionary is nested: first by dataset name, then by split name. The dataset names are stored in [`DatasetInfo`](https://instadeepai.github.io/mlip/api_reference/data/dataset_info.html) and are later resolved by `InferenceContext(dataset_name=...)`.


In [ ]:
readers = {
    "aspirin": {
        "train": ExtxyzReader("mixture_of_experts/aspirin_train.xyz"),
        "val": ExtxyzReader("mixture_of_experts/aspirin_val.xyz"),
    },
    "3bpa": {
        "train": ExtxyzReader("mixture_of_experts/3bpa_train.xyz"),
        "val": ExtxyzReader("mixture_of_experts/3bpa_val.xyz"),
    },
}

builder_config = GraphDatasetBuilder.Config(
    graph_cutoff_angstrom=5.0,
    batch_size=4,
)

builder = GraphDatasetBuilder(readers, builder_config, BuilderMode.MULTI)
splits = builder.get_datasets()

train_set, validation_set = splits["train"], splits["val"]

print("Dataset names:", builder.dataset_info.dataset_name)
print("Number of train batches:", len(train_set))
print("Number of validation batches:", len(validation_set))

## 2. Preparing a MoE model

To start training, we first need to prepare a model architecture. In this tutorial, we use a very small eSEN model and activate its Mixture-of-Experts configuration through `EsenMoEConfig`.

We use linear MoE for expert routing. In this example, `EsenMoEConfig(..., routing_globals=("dataset_idx",))` routes the experts using the graph-level dataset index. More generally one can also use other graph globals such as `spin_multiplicity` and `charge` when they are included in the data (see later sections). 

The multi-dataset builder tags training graphs with `dataset_idx`. At inference time, the graph does not naturally know which dataset it belongs to, so we provide this metadata with `InferenceContext`.


In [ ]:
mlip_network = Esen(
    EsenConfig(
        num_layers=1,
        sphere_channels=8,
        hidden_channels=8,
        edge_channels=8,
        num_rbf=8,
        l_max=1,
        m_max=1,
        moe=EsenMoEConfig(
            num_experts=2,
            routing_globals=("dataset_idx",),
            embed_dim=8,
            router_hidden_dims=[8],
        ),
    ),
    builder.dataset_info,
)

force_field = ForceField.from_mlip_network(mlip_network, seed=42)

## 3. Running a training loop

We can now create and run a standard [`TrainingLoop`](https://instadeepai.github.io/mlip/api_reference/training/training_loop.html). This run is intentionally small and is only meant to produce a saved model that we can reload with different inference contexts.


In [ ]:
training_loop = TrainingLoop(
    train_dataset=train_set,
    validation_dataset=validation_set,
    force_field=force_field,
    loss=MSELoss(),
    optimizer=get_default_mlip_optimizer(),
    config=TrainingLoop.Config(num_epochs=2),
)

training_loop.run()

In [ ]:
optimized_force_field = training_loop.best_model

moe_data_dir = Path("mixture_of_experts")
moe_data_dir.mkdir(exist_ok=True)
model_path = moe_data_dir / "final_moe_model.zip"

save_model_to_zip(model_path, optimized_force_field)

## 4. Loading the same model with different inference contexts

The saved model has two datasets recorded in its `DatasetInfo`. When we load it for inference, we attach the context we want to use. The context resolves the human-readable `dataset_name` to the corresponding `dataset_idx`.

We do this for MoE models because their expert weights depend on graph-level metadata. In eSEN's mixture of linear experts, MoE dense layers store one kernel per expert and the router computes coefficients from global fields such as `dataset_idx`, `charge`, or `spin_multiplicity`. If these global values are known before inference, the model does not need to keep routing dynamically: the expert kernels can be linearly combined once for that fixed context. In the *mlip* library, `load_model_from_zip(..., inference_context=...)` performs this preparation automatically for MoE models and returns a force field where the experts have been contracted for the requested context. After loading an MoE model with an inference context, the parameters have been contracted, so the resulting model is no longer an MoE model.


In [ ]:
aspirin_force_field = load_model_from_zip(
    Esen,
    model_path,
    inference_context=InferenceContext(dataset_name="aspirin"),
)

bpa_force_field = load_model_from_zip(
    Esen,
    model_path,
    inference_context=InferenceContext(dataset_name="3bpa"),
)

print("Aspirin context:", aspirin_force_field.inference_context)
print("3BPA context:", bpa_force_field.inference_context)

print(
    "Aspirin model MoE config after loading:",
    aspirin_force_field.predictor.mlip_network.config.moe,
)
print(
    "3BPA model MoE config after loading:",
    bpa_force_field.predictor.mlip_network.config.moe,
)

Both force fields came from the same zip file and use the same readout. The difference is the inference context attached at loading time, which fixes the MoE routing metadata and therefore the contracted expert parameters. We can now run batched inference with the same helper function used in the simulation tutorial.


In [ ]:
aspirin_structures = ase_read("mixture_of_experts/aspirin_test.xyz", index=":")
bpa_structures = ase_read("mixture_of_experts/3bpa_test.xyz", index=":")

aspirin_predictions = run_batched_inference(
    aspirin_structures,
    aspirin_force_field,
    batch_size=2,
)
bpa_predictions = run_batched_inference(
    bpa_structures,
    bpa_force_field,
    batch_size=2,
)

print("Aspirin energy:", aspirin_predictions[0].energy)
print("3BPA energy:", bpa_predictions[0].energy)

## 5. Advanced topic: Routing on charge and spin

MoE routing can also use other graph-level metadata: `charge` and `spin_multiplicity`. The code snippet below is added for illustrative purposes, as datasets including charges and spin are not convenient to use for a simple tutorial. The routing pattern is the same: during training, each graph includes the relevant global fields. During inference, supplying an `InferenceContext` enables efficient parameter contraction based on those global values.




In [ ]:
# Example only: this is not run in this tutorial.
#
# mlip_network = Esen(
#     EsenConfig(
#         moe=EsenMoEConfig(
#             num_experts=4,
#             routing_globals=("charge", "spin_multiplicity"),
#         ),
#     ),
#     builder.dataset_info,
# )
#
# charged_force_field = load_model_from_zip(
#     Esen,
#     "path/to/moe_model.zip",
#     inference_context=InferenceContext(charge=1, spin_multiplicity=2),
# )
#
# You can also combine all available routing globals:
# routing_globals=("dataset_idx", "charge", "spin_multiplicity")


## 6. Invalid inference contexts

Dataset names are validated against the [`DatasetInfo`](https://instadeepai.github.io/mlip/api_reference/data/dataset_info.html) saved with the model. If a name is unknown, loading fails early with a clear error.


In [ ]:
try:
    load_model_from_zip(
        Esen,
        model_path,
        inference_context=InferenceContext(dataset_name="unknown_dataset"),
    )
except ValueError as exc:
    print(exc)

For production workflows, the important pattern is therefore:

1. Train multi-dataset models with stable dataset names.
2. Save the trained force field.
3. Load with `InferenceContext(...)` for the context you want to deploy.
